Setup Utils

In [0]:
from pyspark.sql.functions import col, when, datediff, current_timestamp, upper, coalesce, lit, round, last, expr, sequence, to_date, explode, row_number
from pyspark.sql.window import Window

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

Funções Auxiliares

In [0]:
def deduplicar(df, col_id):
    windowSpec = Window.partitionBy(col_id).orderBy(col("timestamp_ingestion").desc())
    return df.withColumn("rn", row_number().over(windowSpec)).filter(col("rn") == 1).drop("rn")

Dimensão - Consumidores

In [0]:
df_customers = spark.table("bronze.tb_customers")
df_customers = df_customers.withColumnRenamed("customer_id", "id_consumidor") \
    .withColumnRenamed("customer_unique_id", "id_unico_consumidor") \
    .withColumnRenamed("customer_zip_code_prefix", "prefixo_cep") \
    .withColumnRenamed("customer_name", "nome_consumidor") \
    .withColumnRenamed("customer_city", "cidade") \
    .withColumnRenamed("customer_state", "estado")

df_customers = deduplicar(df_customers, "id_consumidor")
df_customers = df_customers.withColumn("cidade", upper(col("cidade"))) \
    .withColumn("estado", upper(col("estado")))

df_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_consumidores")

Dimensão - Produtos

In [0]:
df_products = spark.table("bronze.tb_products")
df_products = df_products.withColumnRenamed("product_id", "id_produto") \
    .withColumnRenamed("product_name", "nome_produto") \
    .withColumnRenamed("product_category_name", "categoria_produto") \
    .withColumnRenamed("product_name_lenght", "tamanho_nome_produto") \
    .withColumnRenamed("product_description_lenght", "tamanho_descricao_produto") \
    .withColumnRenamed("product_photos_qty", "quantidade_fotos") \
    .withColumnRenamed("product_weight_g", "peso_produto_gramas") \
    .withColumnRenamed("product_length_cm", "comprimento_centimetros") \
    .withColumnRenamed("product_height_cm", "altura_centimetros") \
    .withColumnRenamed("product_width_cm", "largura_centimetros")

df_products = deduplicar(df_products, "id_produto")

df_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_produtos")

Dimensão - Vendedores

In [0]:
df_sellers = spark.table("bronze.tb_sellers")
df_sellers = df_sellers.withColumnRenamed("seller_id", "id_vendedor") \
    .withColumnRenamed("seller_name", "nome_vendedor") \
    .withColumnRenamed("seller_zip_code_prefix", "prefixo_cep") \
    .withColumnRenamed("seller_city", "cidade") \
    .withColumnRenamed("seller_state", "estado")

df_sellers = deduplicar(df_sellers, "id_vendedor")
df_sellers = df_sellers.withColumn("cidade", upper(col("cidade"))).withColumn("estado", upper(col("estado")))

df_sellers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_vendedores")

Dimensão Tradução de Categorias

In [0]:
df_translation = spark.table("bronze.tb_product_category_name_translation")
df_translation = df_translation.withColumnRenamed("product_category_name", "nome_produto_pt") \
    .withColumnRenamed("product_category_name_english", "nome_produto_en")

df_translation.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_categoria_produtos_traducao")

Fato - Pedidos

In [0]:
df_orders = spark.table("bronze.tb_orders")
df_orders = df_orders.withColumnRenamed("order_id", "id_pedido") \
    .withColumnRenamed("customer_id", "id_consumidor") \
    .withColumnRenamed("order_status", "status") \
    .withColumnRenamed("order_purchase_timestamp", "data_compra") \
    .withColumnRenamed("order_approved_at", "data_aprovacao") \
    .withColumnRenamed("order_delivered_carrier_date", "data_envio_transportadora") \
    .withColumnRenamed("order_delivered_customer_date", "data_entrega_real") \
    .withColumnRenamed("order_estimated_delivery_date", "data_entrega_estimada")

# Tradução de Status
df_orders = df_orders.withColumn("status", when(col("status") == "delivered", "entregue")
    .when(col("status") == "canceled", "cancelado")
    .when(col("status") == "shipped", "enviado")
    .when(col("status") == "processing", "processando")
    .when(col("status") == "invoiced", "faturado")
    .when(col("status") == "unavailable", "indisponível")
    .when(col("status") == "created", "criado")
    .when(col("status") == "approved", "aprovado")
    .otherwise(col("status")))

# Colunas Derivadas
df_orders = df_orders.withColumn("tempo_entrega_dias", datediff(col("data_entrega_real"), col("data_compra"))) \
    .withColumn("tempo_entrega_estimado_dias", datediff(col("data_entrega_estimada"), col("data_compra"))) \
    .withColumn("diferenca_entrega_dias", datediff(col("data_entrega_real"), col("data_entrega_estimada"))) \
    .withColumn("entrega_no_prazo", 
                when(col("data_entrega_real").isNull(), "Não Entregue")
                .when(col("diferenca_entrega_dias") <= 0, "Sim")
                .otherwise("Não"))

df_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fat_pedidos")

Fato - Itens dos Pedidos

In [0]:
df_items = spark.table("bronze.tb_order_items")
df_items = df_items.withColumnRenamed("order_id", "id_pedido") \
    .withColumnRenamed("order_item_id", "id_item") \
    .withColumnRenamed("product_id", "id_produto") \
    .withColumnRenamed("seller_id", "id_vendedor") \
    .withColumnRenamed("shipping_limit_date", "data_limite_envio") \
    .withColumnRenamed("price", "preco_BRL") \
    .withColumnRenamed("freight_value", "preco_frete")

df_items.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fat_itens_pedidos")

Fato - Pagamentos

In [0]:
df_payments = spark.table("bronze.tb_order_payments")
df_payments = df_payments.withColumnRenamed("order_id", "id_pedido") \
    .withColumnRenamed("payment_sequential", "sequencial_pagamento") \
    .withColumnRenamed("payment_type", "tipo_pagamento") \
    .withColumnRenamed("payment_installments", "parcelas_pagamento") \
    .withColumnRenamed("payment_value", "valor_pagamento")

df_payments = df_payments.withColumn("tipo_pagamento", 
    when(col("tipo_pagamento") == "credit_card", "Cartão de Crédito")
    .when(col("tipo_pagamento") == "boleto", "Boleto")
    .when(col("tipo_pagamento") == "voucher", "Voucher")
    .when(col("tipo_pagamento") == "debit_card", "Cartão de Débito")
    .otherwise("Não Definido"))

df_payments.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fat_pagamentos_pedidos")

Fato - Avaliações

In [0]:
df_reviews = spark.table("bronze.tb_order_reviews")

df_reviews = df_reviews.withColumnRenamed("review_id", "id_avaliacao") \
    .withColumnRenamed("order_id", "id_pedido") \
    .withColumnRenamed("review_score", "nota_avaliacao") \
    .withColumnRenamed("review_comment_title", "titulo_avaliacao_comentario") \
    .withColumnRenamed("review_comment_message", "mensagem_avaliacao_comentario") \
    .withColumnRenamed("review_creation_date", "data_criacao_avaliacao") \
    .withColumnRenamed("review_answer_timestamp", "data_resposta_avaliacao")

# Tolerância a falhas na data
df_reviews = df_reviews.withColumn("data_criacao_avaliacao", expr("try_to_timestamp(data_criacao_avaliacao)")) \
    .withColumn("data_resposta_avaliacao", expr("try_to_timestamp(data_resposta_avaliacao)"))

df_reviews = df_reviews.filter(col("id_pedido").isNotNull() & (col("data_criacao_avaliacao") <= current_timestamp()))

# Preenchimento explícito de vazios
df_reviews = df_reviews.withColumn("titulo_avaliacao_comentario", coalesce(col("titulo_avaliacao_comentario"), lit("Sem título"))) \
    .withColumn("mensagem_avaliacao_comentario", coalesce(col("mensagem_avaliacao_comentario"), lit("Sem comentário")))

df_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fat_avaliacoes_pedidos")

Calendário e Cotação do Dólar

In [0]:
df_api = spark.table("bronze.tb_cotacao_dolar")
df_api = df_api.withColumn("data_ref", to_date(col("dataHoraCotacao")))

# Gera calendário que cobre o período das compras do dataset (2016 a 2018)
df_calendario = spark.sql("SELECT explode(sequence(to_date('2016-09-01'), to_date('2018-10-31'), interval 1 day)) as data_ref")

# Preenche buracos com a cotação de fechamento anterior
window_cotacao = Window.orderBy("data_ref").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_cotacao_final = df_calendario.join(df_api, "data_ref", "left") \
    .withColumn("cotacaoCompra_preenchida", last("cotacaoCompra", ignorenulls=True).over(window_cotacao)) \
    .select(col("data_ref").alias("data_cotacao"), col("cotacaoCompra_preenchida").alias("valor_cotacao_usd"))

df_cotacao_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_cotacao_dolar")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela Final Silver (Fato Pedido Total)

In [0]:
df_pedidos_silver = spark.table("silver.fat_pedidos").select("id_pedido", "id_consumidor", "status", to_date("data_compra").alias("data_pedido"))

df_pagamentos_agrupados = spark.table("silver.fat_pagamentos_pedidos").groupBy("id_pedido").sum("valor_pagamento").withColumnRenamed("sum(valor_pagamento)", "valor_total_pago_brl")

df_fat_total = df_pedidos_silver.join(df_pagamentos_agrupados, "id_pedido", "left") \
    .join(df_cotacao_final, df_pedidos_silver.data_pedido == df_cotacao_final.data_cotacao, "left")

df_fat_total = df_fat_total.withColumn("valor_total_pago_usd", col("valor_total_pago_brl") / col("valor_cotacao_usd")) \
    .withColumn("valor_total_pago_brl", round(col("valor_total_pago_brl"), 2)) \
    .withColumn("valor_total_pago_usd", round(col("valor_total_pago_usd"), 2)) \
    .select("id_pedido", "id_consumidor", "status", "valor_total_pago_brl", "valor_total_pago_usd", "data_pedido")

df_fat_total.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fat_pedido_total")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
